In [4]:
# 1. INSTALACIÓN DE MÓDULOS (IPOPT viene dentro de 'coin')
%pip install -q amplpy ipywidgets
!python -m amplpy.modules install coin

import pandas as pd
from amplpy import AMPL, modules
from IPython.display import display, HTML
import ipywidgets as widgets

# 2. INICIALIZACIÓN
modules.load() 

ampl = AMPL()
ampl.reset() # Limpia la memoria de la celda

pd.options.display.float_format = '{:,.2f}'.format
pd.set_option('display.max_rows', None)

# 3. TRADUCCIÓN DEL MODELO PYOMO A AMPL (.mod)
ampl.eval(r"""
# --- Parámetros Generados ---
param p1 := 300.96465304963374;
param p2 := 528.3977262282696;
param d1 := 91.68913891717692;
param d2 := 195.17281591389383;
param l := 5;
param alpha := 0.39859135442673926;
param Cp1 := 1225.9169190669074;
param Cp2 := 983.4933511504164;
param S := 313583.5019937311;
param H1 := 8.204894824593927;
param H2 := 6.208876972893994;
param H_mp := 2.9899636246608567;
param C_bo := 49.0345295322654;
param C_vp := 2963.0000489493027;

# --- Variables de Decisión ---
var Q >= 0, := 5000.0;
var I >= 1, := 100.0;
var c >= 1, := 100.0;

# --- Expresiones (Variables Dependientes) ---
# Relaciones P1 (Frutibars)
var B = (alpha * c) / (1 - alpha);
var t1 = B / (p1 - (d1 + alpha * d1));
var t2 = I / (p1 - d1);
var t5 = c / ((1 - alpha) * d1);
var t6 = I / (p2 - d2);

# Relaciones MP
var T_I1 = t1 + t2;
var T_I2 = t6;

var t10 = T_I2;
var t11 = T_I1 - T_I2;
var t12 = (Q - (p1 + p2) * t10 - (p1 + 0.9 * p2) * t11) / (0.8 * p1 + 0.9 * p2);
var tp = t10 + t11 + t12;

# Tiempos y Niveles Restantes
var t3 = tp - T_I1;
var t7 = tp - T_I2;
var Imax1 = I + (0.8 * p1 - d1) * t3;
var Imax2 = I + (0.9 * p2 - d2) * t7;
var t4 = (Imax1 - c) / d1;
var t8 = Imax2 / d2;
var T = tp + t4 + t5;
var t9 = t4 + t5 - t8;

# --- Componentes de Costos ---
param C_S := S; # <--- CORREGIDO: Ahora es un param, no un var.

var Q1 = p1 * T_I1 + 0.8 * p1 * t3;
var Q2 = p2 * T_I2 + 0.9 * p2 * t7;
var C_Prod = Cp1 * Q1 + Cp2 * Q2;

var Q3 = Q - (p1 + p2) * t10;
var Q4 = Q3 - (p1 + 0.9 * p2) * t11;
var Area_MP = (Q + Q3) / 2 * t10 + (Q3 + Q4) / 2 * t11 + (Q4 / 2) * t12;
var C_Hmp = H_mp * Area_MP;

var Area_P1_pos = (I / 2) * t2 + (I + Imax1) / 2 * t3 + (Imax1 + c) / 2 * t4 + (c / 2) * t5;
var C_H1 = H1 * Area_P1_pos;

var Area_P1_neg = (B / 2) * t1 + (B / 2) * t5;
var C_BO1 = C_bo * Area_P1_neg;

var Area_P2_pos = (I / 2) * t6 + (I + Imax2) / 2 * t7 + (Imax2 / 2) * t8;
var C_H2 = H2 * Area_P2_pos;

var Ventas_Perdidas_P2 = d2 * t9;
var C_VP2 = C_vp * Ventas_Perdidas_P2;

var C_ciclo = C_S + C_Prod + C_Hmp + C_H1 + C_BO1 + C_H2 + C_VP2;
var C_diario = C_ciclo / T;

# --- Función Objetivo ---
minimize Objetivo_Diario: C_diario;

# --- Restricciones ---
subject to R1_t3: t3 >= 0;
subject to R2_t7: t7 >= 0;
subject to R3_t12: t12 >= 0;
subject to R4_t9: t9 >= 0;
subject to R5_Supuesto: T_I2 <= T_I1;
""")

# 4. RESOLUCIÓN CON IPOPT
ampl.set_option('solver', 'ipopt')
print("\n--- Ejecutando Optimización No Lineal (IPOPT) ---")
ampl.solve()

# 5. REPORTE VISUAL INTERACTIVO
C_optimo = ampl.get_objective('Objetivo_Diario').value()
T_val = ampl.get_value('T')

header_html = f"""
<div style="background-color: #1B4F72; padding: 25px; border-radius: 12px; color: white; font-family: sans-serif;">
    <h1 style="margin: 0;">Solución Óptima: Gestión de Inventario</h1>
    <hr style="border: 0.5px solid #5DADE2;">
    <p style="font-size: 20px;">Costo Total Diario: <span style="color: #F7DC6F;"><b>$ {C_optimo:,.2f}</b></span></p>
    <p style="font-size: 14px;">Solver: <b>IPOPT</b> | Lead Time Proveedor: <b>{ampl.get_value('l'):.0f} días</b></p>
</div>
"""

# Agrupando datos en DataFrames para las pestañas
df_decisiones = pd.DataFrame([
    ("Q (Lote Materia Prima)", ampl.get_value('Q')),
    ("I (Nivel desaceleración)", ampl.get_value('I')),
    ("c (Nivel racionamiento P1)", ampl.get_value('c'))
], columns=['Variable', 'Valor [unidades]']).set_index('Variable')

df_inventarios = pd.DataFrame([
    ("Imax1 (Máx. Frutibars)", ampl.get_value('Imax1')),
    ("Imax2 (Máx. Frutipack)", ampl.get_value('Imax2')),
    ("B (Máx. Backorder P1)", ampl.get_value('B')),
    ("Inv. Promedio MP", ampl.get_value('Area_MP') / T_val),
    ("Inv. Promedio P1 (on-hand)", ampl.get_value('Area_P1_pos') / T_val),
    ("Backorder Promedio P1", ampl.get_value('Area_P1_neg') / T_val),
    ("Inv. Promedio P2 (on-hand)", ampl.get_value('Area_P2_pos') / T_val)
], columns=['Métrica', 'Valor [unid o unid/día]']).set_index('Métrica')

df_tiempos = pd.DataFrame([
    ("T (Ciclo Total)", T_val),
    ("tp (Producción Total)", ampl.get_value('tp')),
    ("t1 (Cubrir Backorder P1)", ampl.get_value('t1')),
    ("t2 (P1 de 0 a I)", ampl.get_value('t2')),
    ("t3 (P1 de I a Imax1)", ampl.get_value('t3')),
    ("t4 (P1 de Imax1 a c)", ampl.get_value('t4')),
    ("t5 (P1 de c a 0)", ampl.get_value('t5')),
    ("t6 (P2 de 0 a I)", ampl.get_value('t6')),
    ("t7 (P2 de I a Imax2)", ampl.get_value('t7')),
    ("t8 (P2 de Imax2 a 0)", ampl.get_value('t8')),
    ("t9 (P2 Venta Perdida)", ampl.get_value('t9')),
    ("t10 (MP - Fase 1)", ampl.get_value('t10')),
    ("t11 (MP - Fase 2)", ampl.get_value('t11')),
    ("t12 (MP - Fase 3)", ampl.get_value('t12'))
], columns=['Segmento', 'Valor [días]']).set_index('Segmento')

df_costos = pd.DataFrame([
    ("Costo Pedido (S) diario", ampl.get_value('C_S') / T_val),
    ("Costo Producción diario", ampl.get_value('C_Prod') / T_val),
    ("Costo Almacén MP diario", ampl.get_value('C_Hmp') / T_val),
    ("Costo Almacén P1 diario", ampl.get_value('C_H1') / T_val),
    ("Costo Backorder P1 diario", ampl.get_value('C_BO1') / T_val),
    ("Costo Almacén P2 diario", ampl.get_value('C_H2') / T_val),
    ("Costo Venta Perdida P2 diario", ampl.get_value('C_VP2') / T_val)
], columns=['Componente', 'Costo [$]']).set_index('Componente')

# Función de color
def color_zeros(val):
    color = '#D3D3D3' if abs(val) < 1e-6 else 'black'
    return f'color: {color}'

# Creación de Pestañas
dataframes = {
    'Decisiones (Q, I, c)': df_decisiones,
    'Tiempos del Ciclo': df_tiempos,
    'Niveles de Inventario': df_inventarios,
    'Desglose de Costos': df_costos
}

tabs = []
for nombre, df in dataframes.items():
    styler = df.style.applymap(color_zeros).format("{:,.2f}")
    if nombre in ['Tiempos del Ciclo', 'Desglose de Costos']:
        styler = styler.bar(subset=[df.columns[0]], color='#AED6F1')
        
    out = widgets.Output()
    with out:
        display(HTML(f"<h3>Detalle: {nombre}</h3>"))
        display(styler)
    tabs.append(out)

tab_control = widgets.Tab(children=tabs)
for i, nombre in enumerate(dataframes.keys()):
    tab_control.set_title(i, nombre)

# Renderizar
display(HTML(header_html))
display(tab_control)

$ /usr/bin/python3 -m pip install -i https://pypi.ampl.com ampl_module_base ampl_module_coin
Looking in indexes: https://pypi.ampl.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 35.4 MB/s eta 0:00:00
Imported ampl_module_base.
Imported ampl_module_base.
Imported ampl_module_coin.

--- Ejecutando Optimización No Lineal (IPOPT) ---
Ipopt 3.12.13: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt
******************************************************************************

This is Ipopt version 3.12.13, running with linear solver mumps.
NOTE: Other linear solvers might be more efficient (see Ipopt documentation).

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constra

/tmp/ipykernel_4598/1270566530.py:184: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styler = df.style.applymap(color_zeros).format("{:,.2f}")


/tmp/ipykernel_4598/1270566530.py:184: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styler = df.style.applymap(color_zeros).format("{:,.2f}")


In [5]:
# 1. INSTALACIÓN DE MÓDULOS (IPOPT viene dentro de 'coin')
# (Si ya corriste esto en tu sesión, puedes omitirlo)
%pip install -q amplpy ipywidgets
!python -m amplpy.modules install coin

import pandas as pd
from amplpy import AMPL, modules
from IPython.display import display, HTML
import ipywidgets as widgets

# 2. INICIALIZACIÓN
modules.load() 

ampl = AMPL()
ampl.reset() # Limpia la memoria de la celda

pd.options.display.float_format = '{:,.2f}'.format
pd.set_option('display.max_rows', None)

# 3. TRADUCCIÓN DEL MODELO PYOMO A AMPL (.mod) - SITUACIÓN 2
ampl.eval(r"""
# --- Parámetros Originales ---
param p1 := 300.96465304963374;
param p2 := 528.3977262282696;
param d1 := 91.68913891717692;
param d2 := 195.17281591389383;
param l := 5;
param alpha := 0.39859135442673926;
param Cp1 := 1225.9169190669074;
param Cp2 := 983.4933511504164;
param S := 313583.5019937311;
param H1 := 8.204894824593927;
param H2 := 6.208876972893994;
param H_mp := 2.9899636246608567;
param C_bo := 49.0345295322654;
param C_vp := 2963.0000489493027;

# --- Nuevos Parámetros (Situación 2) ---
param Q_fijo := 1600.0;
param Imax1_limite := 360.0;

# --- Variables de Decisión (MODIFICADAS) ---
# Q ya no es variable. Solo quedan I y c.
var I >= 1, := 100.0;
var c >= 1, := 100.0;

# --- Expresiones (Variables Dependientes) ---
# Relaciones P1 (Frutibars)
var B = (alpha * c) / (1 - alpha);
var t1 = B / (p1 - (d1 + alpha * d1));
var t2 = I / (p1 - d1);
var t5 = c / ((1 - alpha) * d1);
var t6 = I / (p2 - d2);

# Relaciones MP
var T_I1 = t1 + t2;
var T_I2 = t6;

var t10 = T_I2;
var t11 = T_I1 - T_I2;

# MODIFICADO: Uso de Q_fijo
var t12 = (Q_fijo - (p1 + p2) * t10 - (p1 + 0.9 * p2) * t11) / (0.8 * p1 + 0.9 * p2);
var tp = t10 + t11 + t12;

# Tiempos y Niveles Restantes
var t3 = tp - T_I1;
var t7 = tp - T_I2;
var Imax1 = I + (0.8 * p1 - d1) * t3;
var Imax2 = I + (0.9 * p2 - d2) * t7;
var t4 = (Imax1 - c) / d1;
var t8 = Imax2 / d2;
var T = tp + t4 + t5;
var t9 = t4 + t5 - t8;

# --- Componentes de Costos ---
param C_S := S; 

var Q1 = p1 * T_I1 + 0.8 * p1 * t3;
var Q2 = p2 * T_I2 + 0.9 * p2 * t7;
var C_Prod = Cp1 * Q1 + Cp2 * Q2;

# MODIFICADO: Uso de Q_fijo
var Q3 = Q_fijo - (p1 + p2) * t10;
var Q4 = Q3 - (p1 + 0.9 * p2) * t11;
var Area_MP = (Q_fijo + Q3) / 2 * t10 + (Q3 + Q4) / 2 * t11 + (Q4 / 2) * t12;
var C_Hmp = H_mp * Area_MP;

var Area_P1_pos = (I / 2) * t2 + (I + Imax1) / 2 * t3 + (Imax1 + c) / 2 * t4 + (c / 2) * t5;
var C_H1 = H1 * Area_P1_pos;

var Area_P1_neg = (B / 2) * t1 + (B / 2) * t5;
var C_BO1 = C_bo * Area_P1_neg;

var Area_P2_pos = (I / 2) * t6 + (I + Imax2) / 2 * t7 + (Imax2 / 2) * t8;
var C_H2 = H2 * Area_P2_pos;

var Ventas_Perdidas_P2 = d2 * t9;
var C_VP2 = C_vp * Ventas_Perdidas_P2;

var C_ciclo = C_S + C_Prod + C_Hmp + C_H1 + C_BO1 + C_H2 + C_VP2;
# Se agrega 1e-9 para protección matemática, igual que en tu Pyomo
var C_diario = C_ciclo / (T + 1e-9);

# --- Función Objetivo ---
minimize Objetivo_Diario: C_diario;

# --- Restricciones ---
subject to R1_t3: t3 >= 0;
subject to R2_t7: t7 >= 0;
subject to R3_t12: t12 >= 0;
subject to R4_t9: t9 >= 0;
subject to R5_Supuesto: T_I2 <= T_I1;
# NUEVA RESTRICCIÓN (SITUACIÓN 2)
subject to R6_Limite_Imax1: Imax1 <= Imax1_limite;
""")

# 4. RESOLUCIÓN CON IPOPT
ampl.set_option('solver', 'ipopt')
print("\n--- Ejecutando Optimización No Lineal (IPOPT) ---")
ampl.solve()

# 5. REPORTE VISUAL INTERACTIVO
C_optimo = ampl.get_objective('Objetivo_Diario').value()
T_val = ampl.get_value('T')
Q_fijo = ampl.get_value('Q_fijo')

header_html = f"""
<div style="background-color: #1B4F72; padding: 25px; border-radius: 12px; color: white; font-family: sans-serif;">
    <h1 style="margin: 0;">Solución Óptima: Situación 2</h1>
    <hr style="border: 0.5px solid #5DADE2;">
    <p style="font-size: 20px;">Costo Total Diario: <span style="color: #F7DC6F;"><b>$ {C_optimo:,.2f}</b></span></p>
    <p style="font-size: 14px;">Solver: <b>IPOPT</b> | Q (MP): <b>Fijo en {Q_fijo:,.0f} unds</b> | Límite Imax1: <b>360 unds</b></p>
</div>
"""

# DataFrames para las pestañas
df_decisiones = pd.DataFrame([
    ("I (Nivel desaceleración)", ampl.get_value('I')),
    ("c (Nivel racionamiento P1)", ampl.get_value('c'))
], columns=['Variable de Decisión', 'Valor [unidades]']).set_index('Variable de Decisión')

df_inventarios = pd.DataFrame([
    ("Imax1 (Máx. Frutibars)", ampl.get_value('Imax1')),
    ("Imax2 (Máx. Frutipack)", ampl.get_value('Imax2')),
    ("B (Máx. Backorder P1)", ampl.get_value('B')),
    ("Inv. Promedio MP", ampl.get_value('Area_MP') / (T_val + 1e-9)),
    ("Inv. Promedio P1 (on-hand)", ampl.get_value('Area_P1_pos') / (T_val + 1e-9)),
    ("Backorder Promedio P1", ampl.get_value('Area_P1_neg') / (T_val + 1e-9)),
    ("Inv. Promedio P2 (on-hand)", ampl.get_value('Area_P2_pos') / (T_val + 1e-9))
], columns=['Métrica', 'Valor [unid o unid/día]']).set_index('Métrica')

df_tiempos = pd.DataFrame([
    ("T (Ciclo Total)", T_val),
    ("tp (Producción Total)", ampl.get_value('tp')),
    ("t1 (Cubrir Backorder P1)", ampl.get_value('t1')),
    ("t2 (P1 de 0 a I)", ampl.get_value('t2')),
    ("t3 (P1 de I a Imax1)", ampl.get_value('t3')),
    ("t4 (P1 de Imax1 a c)", ampl.get_value('t4')),
    ("t5 (P1 de c a 0)", ampl.get_value('t5')),
    ("t6 (P2 de 0 a I)", ampl.get_value('t6')),
    ("t7 (P2 de I a Imax2)", ampl.get_value('t7')),
    ("t8 (P2 de Imax2 a 0)", ampl.get_value('t8')),
    ("t9 (P2 Venta Perdida)", ampl.get_value('t9')),
    ("t10 (MP - Fase 1)", ampl.get_value('t10')),
    ("t11 (MP - Fase 2)", ampl.get_value('t11')),
    ("t12 (MP - Fase 3)", ampl.get_value('t12'))
], columns=['Segmento', 'Valor [días]']).set_index('Segmento')

df_costos = pd.DataFrame([
    ("Costo Pedido (S) diario", ampl.get_value('C_S') / (T_val + 1e-9)),
    ("Costo Producción diario", ampl.get_value('C_Prod') / (T_val + 1e-9)),
    ("Costo Almacén MP diario", ampl.get_value('C_Hmp') / (T_val + 1e-9)),
    ("Costo Almacén P1 diario", ampl.get_value('C_H1') / (T_val + 1e-9)),
    ("Costo Backorder P1 diario", ampl.get_value('C_BO1') / (T_val + 1e-9)),
    ("Costo Almacén P2 diario", ampl.get_value('C_H2') / (T_val + 1e-9)),
    ("Costo Venta Perdida P2 diario", ampl.get_value('C_VP2') / (T_val + 1e-9))
], columns=['Componente', 'Costo [$]']).set_index('Componente')

# Función de color
def color_zeros(val):
    color = '#D3D3D3' if abs(val) < 1e-6 else 'black'
    return f'color: {color}'

# Creación de Pestañas
dataframes = {
    'Decisiones (I, c)': df_decisiones,
    'Tiempos del Ciclo': df_tiempos,
    'Niveles de Inventario': df_inventarios,
    'Desglose de Costos': df_costos
}

tabs = []
for nombre, df in dataframes.items():
    styler = df.style.applymap(color_zeros).format("{:,.2f}")
    if nombre in ['Tiempos del Ciclo', 'Desglose de Costos']:
        styler = styler.bar(subset=[df.columns[0]], color='#AED6F1')
        
    out = widgets.Output()
    with out:
        display(HTML(f"<h3>Detalle: {nombre}</h3>"))
        display(styler)
    tabs.append(out)

tab_control = widgets.Tab(children=tabs)
for i, nombre in enumerate(dataframes.keys()):
    tab_control.set_title(i, nombre)

# Renderizar Todo
display(HTML(header_html))
display(tab_control)

$ /usr/bin/python3 -m pip install -i https://pypi.ampl.com ampl_module_base ampl_module_coin
Looking in indexes: https://pypi.ampl.com
Imported ampl_module_base.
Imported ampl_module_base.
Imported ampl_module_coin.

--- Ejecutando Optimización No Lineal (IPOPT) ---
Ipopt 3.12.13: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt
******************************************************************************

This is Ipopt version 3.12.13, running with linear solver mumps.
NOTE: Other linear solvers might be more efficient (see Ipopt documentation).

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:       12
Number of nonzeros in Lagrangian Hessian.............:   

/tmp/ipykernel_4598/2860065070.py:195: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styler = df.style.applymap(color_zeros).format("{:,.2f}")


/tmp/ipykernel_4598/2860065070.py:195: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styler = df.style.applymap(color_zeros).format("{:,.2f}")
